# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# The queue is rebuilt here from scratch rather than reloaded, so every number in
# this playbook traces back to code you can re-run. Same eligibility gate and same
# grouped split as ML-09 -- if this cell disagrees with that notebook, one of them
# is wrong and I want to see it.
import json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

OUT = Path("../outputs"); OUT.mkdir(parents=True, exist_ok=True)
FIG = Path("../figures"); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

elig = df[(df.impressions_90d >= 250) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
tier_med = elig.groupby("position_tier")["ctr"].transform("median")
elig["tier_median_ctr"] = tier_med
elig["ctr_gap_ratio"] = ((tier_med - elig.ctr) / tier_med.replace(0, np.nan)).clip(lower=0).fillna(0)
elig["freshness_term"] = (elig.days_since_last_update / 365).clip(upper=1)

FEATURES = ["ctr_gap_ratio", "freshness_term", "ctr", "avg_position", "impressions_90d",
            "clicks_90d", "days_since_last_update", "content_age_days", "word_count",
            "engagement_rate", "scroll_rate", "days_with_impressions", "search_volume",
            "competition"]

# The label trap from the data skill: is_declining comes from trend_direction, which
# comes from trend_pct. Both are permanently banned as features. Asserted, not trusted.
BANNED = {"trend_direction", "trend_pct", "is_declining"}
assert not (set(FEATURES) & BANNED), "a label-derived column got into the feature list"

X = elig[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = elig.is_declining.values
groups = elig.client_id

tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=42).split(X, y, groups))
assert not (set(elig.iloc[tr].client_id) & set(elig.iloc[te].client_id)), "clients leaked across the split"

gb = GradientBoostingClassifier(random_state=42).fit(X.iloc[tr], y[tr])
scores = gb.predict_proba(X.iloc[te])[:, 1]

q = elig.iloc[te].copy()
q["model_score"] = scores
BASE_RATE = float(y[te].mean())
print(f"held-out clients: {q.client_id.nunique()} | pages: {len(q):,} | base rate: {BASE_RATE:.3f}")

held-out clients: 9 | pages: 2,955 | base rate: 0.551


### Archetypes, and why the queue needs them

A score of 0.93 tells a writer nothing they can act on. So every page also gets an
**archetype** — a plain pattern someone can confirm by looking at the page for a
minute — and each archetype maps to one action and one sentence of reasoning.

The order of the rules matters, and the first one is the important one: any page
with zero clicks is caught before anything else can claim it. That population was
the entire false-alarm group in ML-08 and again on a different split in ML-09, and
it climbs the queue for a mechanical reason — a page with no clicks gets the maximum
possible CTR-gap, so the ranking mistakes broken measurement for a content problem.


In [2]:
# --- archetypes -> actions -------------------------------------------------
# The model gives one number. A number is not something a writer can act on, so
# each page also gets an archetype: a plain-language pattern a human can check
# against the page in about a minute. Order matters -- the first rule that fits
# wins, and the no-go check is deliberately first so it can never be overruled.

# Thresholds come from the data, not from a round number that sounded right. An
# earlier version of this cell used "word_count < 600" and "not updated in a
# year". Both matched ZERO rows -- the shortest eligible page is 692 words and the
# oldest update is 301 days. Dead rules that look thorough are worse than no
# rules, so the threshold is computed and the guard below asserts every archetype
# actually fires.
WC_P10 = float(elig.word_count.quantile(0.10))
print(f"shortest-10% word-count threshold: {WC_P10:,.0f} words")


def archetype(r):
    # A page with zero clicks in 90 days has no click trend that can decline. It
    # also gets the maximum ctr_gap_ratio for having no clicks at all, which is how
    # it climbs the queue. This was the whole false-alarm population in ML-08 and
    # again in ML-09, so it is caught first and routed away from refresh work.
    if r.clicks_90d == 0:
        return "zero_click_ghost"
    if r.ctr_gap_ratio >= 0.5 and r.impressions_90d >= 500:
        return "visible_but_unclicked"
    if 11 <= r.avg_position <= 20:
        return "stuck_on_page_two"
    # word_count is missing for 4,081 eligible pages, and missingness tracks
    # content_type -- so "we don't know how long this page is" is its own bucket
    # rather than something quietly filled with a zero.
    if pd.isna(r.word_count):
        return "length_unknown"
    if r.word_count <= WC_P10:
        return "short_for_its_class"
    return "no_clear_pattern"


ACTIONS = {
    "zero_click_ghost":      ("DO NOT REFRESH - check tracking first",
                              "90 days of impressions and zero clicks usually means broken click tracking, a "
                              "bot-heavy query, or a page that answers in the snippet. Refreshing copy fixes none of those."),
    "visible_but_unclicked": ("Rewrite title and meta description",
                              "People see it and skip it. The listing is losing the click, not the page."),
    "stuck_on_page_two":     ("Improve depth and internal links",
                              "Positions 11-20 get almost no clicks; moving up is worth more than any CTR tweak."),
    "length_unknown":        ("Analyst check - no word count on file",
                              "Length is missing for this page, and missingness tracks content_type, so a "
                              "writer would be guessing. Find out what type it is before deciding anything."),
    "short_for_its_class":   ("Expand coverage",
                              "In the shortest 10% of pages that do have a word count on file."),
    "no_clear_pattern":      ("Manual look, no preset action",
                              "The model ranked it but no archetype fits. Treat the score as a hint only."),
}

q["archetype"] = q.apply(archetype, axis=1)
q["action"] = q.archetype.map(lambda a: ACTIONS[a][0])
q["reason_code"] = q.archetype
q["why"] = q.archetype.map(lambda a: ACTIONS[a][1])

# Evidence a reviewer can check without opening the notebook.
q["evidence"] = q.apply(
    lambda r: (f"{int(r.impressions_90d):,} impressions, {int(r.clicks_90d)} clicks, "
               f"CTR {r.ctr:.2f}% vs tier median {r.tier_median_ctr:.2f}%, "
               f"position {r.avg_position:.1f}, {int(r.days_since_last_update)} days since update"), axis=1)

queue = (q.sort_values("model_score", ascending=False)
           .reset_index(drop=True)
           .assign(queue_rank=lambda d: d.index + 1))

unused = set(ACTIONS) - set(queue.archetype.unique())
assert not unused, f"archetype rules that never fire: {unused}"

print(queue.archetype.value_counts().to_string())
print()
print("top 10 of the ranked queue:")
print(queue.head(10)[["queue_rank", "reason_code", "action", "model_score", "evidence"]].to_string(index=False))

shortest-10% word-count threshold: 1,690 words
archetype
no_clear_pattern         1269
stuck_on_page_two         635
zero_click_ghost          388
visible_but_unclicked     362
length_unknown            215
short_for_its_class        86

top 10 of the ranked queue:
 queue_rank           reason_code                                action  model_score                                                                                          evidence
          1 visible_but_unclicked    Rewrite title and meta description     0.968618   2,324 impressions, 1 clicks, CTR 0.04% vs tier median 0.19%, position 0.6, 15 days since update
          2      no_clear_pattern         Manual look, no preset action     0.968069  2,846 impressions, 4 clicks, CTR 0.14% vs tier median 0.19%, position 2.0, 106 days since update
          3 visible_but_unclicked    Rewrite title and meta description     0.959513  1,620 impressions, 1 clicks, CTR 0.06% vs tier median 0.19%, position 1.1, 106 days since update
  

In [3]:
# --- does the ranking actually beat picking at random? ---------------------
# Precision@K on its own is a number with no meaning. Next to the base rate it
# becomes a decision: is a reviewer's hour better spent on this queue than on a
# random 50 pages from the same eligible pool?
def p_at_k(frame, k):
    return float(frame.head(k).is_declining.mean())

assert p_at_k(pd.DataFrame({"is_declining": [1, 0, 1]}), 2) == 0.5, "metric is wrong, stop here"

rows = []
for k in (10, 20, 50, 100, 200):
    p = p_at_k(queue, k)
    rows.append(dict(K=k, precision_at_K=round(p, 3), base_rate=round(BASE_RATE, 3),
                     lift=round(p / BASE_RATE, 2), extra_true_finds=round((p - BASE_RATE) * k, 1)))
prec = pd.DataFrame(rows)
print(prec.to_string(index=False))
print()
print(f"ROC-AUC {roc_auc_score(y[te], scores):.3f} | PR-AUC {average_precision_score(y[te], scores):.3f}")
print()
print("Read the last column as: reviewing this many pages off the top of the queue finds")
print("that many more genuinely declining pages than reviewing the same number at random.")
print("The lift is real but modest -- this is a triage aid, not an oracle.")

  K  precision_at_K  base_rate  lift  extra_true_finds
 10           0.900      0.551  1.63               3.5
 20           0.900      0.551  1.63               7.0
 50           0.880      0.551  1.60              16.5
100           0.780      0.551  1.42              22.9
200           0.735      0.551  1.33              36.9

ROC-AUC 0.618 | PR-AUC 0.667

Read the last column as: reviewing this many pages off the top of the queue finds
that many more genuinely declining pages than reviewing the same number at random.
The lift is real but modest -- this is a triage aid, not an oracle.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This is a **triage aid for a content strategist**, not a decision system.

It answers one question: *given a limited number of review hours this month, which
pages in this client's portfolio are worth looking at first?* It does not answer
"will refreshing this recover traffic" — nothing here measures that, because nobody
ran an experiment where pages were assigned to be refreshed.

**Where it stops being valid**

- Outside the eligibility gate (250+ impressions, a real average position 1–20).
- Across clients — history depth and tracking setup differ, so scores compare only
  inside one client.
- Beyond one 90-day window. The label is cut from that window; a queue from six
  months ago describes a portfolio that no longer exists.
- On any page whose clicks are not being measured correctly, which is the failure
  mode the no-go list covers.

### The decay insight is a negative result, and that is the honest version

I set out to show that the longer a page goes without an update, the more likely it
is to be declining — the finding that would justify a refresh schedule. It is not
in this data, and the cells below show why rather than dressing it up. Writing down
what I expected and did not find is worth more than a chart built on seventeen
pages from one client.


In [4]:
# --- the decay / refresh insight, and why it is a negative result ---------
# The plan was a decay curve: the longer since a page was updated, the more likely
# it is declining. Before drawing it I looked at how the staleness column is
# actually distributed, and the plan did not survive.
vc = elig.days_since_last_update.value_counts().head(4)
print("most common values of days_since_last_update:")
print(vc.to_string())
print()
print(f"top 2 values alone cover {vc.head(2).sum() / len(elig):.0%} of eligible pages")
print()

per_client = elig.groupby("client_id").days_since_last_update.agg(["size", "nunique"])
print("distinct staleness values per client:")
print(per_client["nunique"].describe()[["min", "50%", "max"]].round(1).to_string())
print()
# Ids are pseudonyms but still client data, so they are relabelled before printing.
# The shape is the point; which client it is, is not.
shown = per_client.sort_values("size", ascending=False).head(5).copy()
shown.index = [f"client {chr(65+i)}" for i in range(len(shown))]
print(shown.to_string())
print()
print("One client has 4,760 pages and 3 distinct values. The median client has 3.")
print("A per-page 'last edited' date does not look like this. This column behaves like")
print("a per-client crawl or snapshot date, so comparing pages on it mostly compares")
print("which client they belong to.")

most common values of days_since_last_update:
days_since_last_update
20     4790
104    4755
22     1520
25      374

top 2 values alone cover 70% of eligible pages

distinct staleness values per client:
min     1.0
50%     3.0
max    10.0

          size  nunique
client A  4760        3
client B  1691        9
client C  1595       10
client D  1233        8
client E   646        5

One client has 4,760 pages and 3 distinct values. The median client has 3.
A per-page 'last edited' date does not look like this. This column behaves like
a per-client crawl or snapshot date, so comparing pages on it mostly compares
which client they belong to.


In [5]:
# What that does to the decay claim, measured rather than asserted.
bins = [0, 30, 90, 180, 10**6]
labels = ["0-1 mo", "1-3 mo", "3-6 mo", "6 mo+"]
elig["staleness"] = pd.cut(elig.days_since_last_update, bins=bins, labels=labels, right=False)

decay = (elig.groupby("staleness", observed=True)
              .agg(pages=("is_declining", "size"),
                   declining_rate=("is_declining", "mean"),
                   clients=("client_id", "nunique"))
              .round(3))
print(decay.to_string())
print()
print("Read the 'clients' column before the rate. The 6-month bucket is 17 pages from")
print("1 client, and its 0.82 declining rate is one client's month, not a trend. The")
print("headline I wanted -- 'staleness predicts decline' -- is not supported here.")
print()
print("Stated honestly: in this dataset I could not measure content decay, because the")
print("only staleness column available is effectively constant within a client. The")
print("range is also too short -- 4 to 301 days, so nothing older than ten months exists")
print("to compare against. Answering it needs per-page edit timestamps from the")
print("warehouse release, which is a different query than this snapshot.")
print()
print("This also flags a feature: freshness_term is built from the same column, so in a")
print("split that holds out whole clients it carries almost no usable signal. Keeping it")
print("is harmless, but it should not be described as a freshness signal in the paper.")

           pages  declining_rate  clients
staleness                                
0-1 mo      8632           0.599       27
1-3 mo        85           0.541        6
3-6 mo      4828           0.631       18
6 mo+         17           0.824        6

Read the 'clients' column before the rate. The 6-month bucket is 17 pages from
1 client, and its 0.82 declining rate is one client's month, not a trend. The
headline I wanted -- 'staleness predicts decline' -- is not supported here.

Stated honestly: in this dataset I could not measure content decay, because the
only staleness column available is effectively constant within a client. The
range is also too short -- 4 to 301 days, so nothing older than ten months exists
to compare against. Answering it needs per-page edit timestamps from the
warehouse release, which is a different query than this snapshot.

This also flags a feature: freshness_term is built from the same column, so in a
split that holds out whole clients it carries almost 

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Single hue plus hatching rather than a colour scale: the bars differ in one
# ordered quantity, and an earlier two-colour choice failed a palette check for
# normal-vision separation, so pattern carries the difference instead.
# The chart's job here is to show the weakness, not hide it -- so each bar is
# labelled with how many CLIENTS it rests on, which is what makes the last bar
# unusable.
fig, ax = plt.subplots(figsize=(7.4, 4.2), dpi=140)
bars = ax.bar(decay.index.astype(str), decay.declining_rate,
              color="#3b5bdb", edgecolor="white", hatch="//", linewidth=1.1)
ax.axhline(BASE_RATE, color="#222", linestyle="--", linewidth=1.2)
ax.text(len(decay) - 0.45, BASE_RATE + 0.01, f"overall rate {BASE_RATE:.3f}",
        ha="right", fontsize=9, color="#222")
for b, n, c in zip(bars, decay.pages, decay.clients):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.008,
            f"n={n:,}" + chr(10) + f"{c} client{'s' if c != 1 else ''}", ha="center", fontsize=8, color="#444")
ax.set_ylabel("share of pages measured as declining")
ax.set_xlabel("time since last recorded update")
ax.set_title("No usable decay signal: the staleness column is near-constant per client",
             fontsize=11)
ax.set_ylim(0, max(decay.declining_rate) * 1.30)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "w07_decay_by_staleness.png")
print("saved", FIG / "w07_decay_by_staleness.png")
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 4.0), dpi=140)
ax.plot(prec.K, prec.precision_at_K, marker="o", color="#3b5bdb", linewidth=2, label="queue precision@K")
ax.axhline(BASE_RATE, color="#222", linestyle="--", linewidth=1.2, label=f"base rate {BASE_RATE:.3f}")
for k, pv in zip(prec.K, prec.precision_at_K):
    ax.annotate(f"{pv:.2f}", (k, pv), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_xlabel("pages reviewed off the top of the queue (K)")
ax.set_ylabel("share that really were declining")
ax.set_title("The queue beats random picking, and the edge shrinks as you go deeper", fontsize=11)
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "w07_precision_at_k.png")
print("saved", FIG / "w07_precision_at_k.png")
plt.close(fig)

saved ..\figures\w07_decay_by_staleness.png
saved ..\figures\w07_precision_at_k.png


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A person has to sign off before any page is touched. Written below as
runnable checks rather than a paragraph, because a paragraph is a rule nobody can
test and everybody skips.


In [7]:
# --- what a human must confirm before any page is touched -----------------
# Written as executable checks so the rule is testable, not a paragraph nobody reads.
REVIEW_GATES = [
    ("tracking is alive",   "the page recorded at least one click in the window",
     lambda r: r.clicks_90d > 0),
    ("demand is real",      "at least 500 impressions, so the decline is not noise on a handful of views",
     lambda r: r.impressions_90d >= 500),
    ("position is known",   "avg_position is a real rank, not the 0 that means 'no data'",
     lambda r: r.avg_position > 0),
    ("gap is measurable",   "CTR sits below its own position tier's median",
     lambda r: r.ctr < r.tier_median_ctr),
]

for name, _, fn in REVIEW_GATES:
    queue[f"gate_{name.split()[0]}"] = queue.apply(fn, axis=1)

gate_cols = [c for c in queue.columns if c.startswith("gate_")]
queue["gates_passed"] = queue[gate_cols].sum(axis=1)
queue["review_status"] = np.where(queue.archetype == "zero_click_ghost", "BLOCKED - no-go",
                          np.where(queue.gates_passed == len(REVIEW_GATES), "ready for writer",
                                   "needs an analyst look first"))

print("what a reviewer must confirm on every page before work starts:\n")
for name, why, _ in REVIEW_GATES:
    print(f"  - {name:18s} {why}")
print()
print(queue.review_status.value_counts().to_string())
print()
print("top 50 by review status:")
print(queue.head(50).review_status.value_counts().to_string())

what a reviewer must confirm on every page before work starts:

  - tracking is alive  the page recorded at least one click in the window
  - demand is real     at least 500 impressions, so the decline is not noise on a handful of views
  - position is known  avg_position is a real rank, not the 0 that means 'no data'
  - gap is measurable  CTR sits below its own position tier's median

review_status
needs an analyst look first    1624
ready for writer                943
BLOCKED - no-go                 388

top 50 by review status:
review_status
BLOCKED - no-go                24
ready for writer               21
needs an analyst look first     5


In [8]:
# --- the no-go list -------------------------------------------------------
# Stated as things that must NOT be automated. Each one has a reason from a
# measurement in this project, not a general caution.
NO_GO = [
    ("Publishing any rewrite without a person reading it",
     "The model never reads the page. It ranks on traffic shape alone, so it cannot tell a "
     "page that is genuinely stale from one that is short because the answer is short."),
    ("Acting on zero-click pages",
     f"{int((queue.archetype=='zero_click_ghost').sum())} pages in the held-out set have impressions and "
     "zero clicks. They score high because a zero click count maxes out the CTR-gap term, and they were "
     "the entire false-alarm population in ML-08 and again in ML-09. Refreshing copy cannot fix broken tracking."),
    ("Deleting or de-indexing anything",
     "Nothing in this analysis measures whether a page has value off search -- internal links, "
     "email, sales use. A low score is not evidence of no value."),
    ("Ranking one client against another",
     "Clients differ in history depth and tracking setup. Scores are only comparable inside one client."),
    ("Promising a traffic outcome from a refresh",
     "This is one snapshot with no intervention. Nobody randomly assigned pages to be refreshed, so "
     "the data cannot support 'refreshing this will recover traffic' -- only 'this looks worth a look first'."),
    ("Running it on clients outside the eligibility gate",
     "The gate needs 250+ impressions and a real position. A third of clients in the warehouse have "
     "too little history to qualify; scoring them anyway produces confident nonsense."),
]
for i, (what, why) in enumerate(NO_GO, 1):
    print(f"{i}. {what}\n   why: {why}\n")

1. Publishing any rewrite without a person reading it
   why: The model never reads the page. It ranks on traffic shape alone, so it cannot tell a page that is genuinely stale from one that is short because the answer is short.

2. Acting on zero-click pages
   why: 388 pages in the held-out set have impressions and zero clicks. They score high because a zero click count maxes out the CTR-gap term, and they were the entire false-alarm population in ML-08 and again in ML-09. Refreshing copy cannot fix broken tracking.

3. Deleting or de-indexing anything
   why: Nothing in this analysis measures whether a page has value off search -- internal links, email, sales use. A low score is not evidence of no value.

4. Ranking one client against another
   why: Clients differ in history depth and tracking setup. Scores are only comparable inside one client.

5. Promising a traffic outcome from a refresh
   why: This is one snapshot with no intervention. Nobody randomly assigned pages to be refr

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Two questions: what would tell me these recommendations went stale, and how
small a change is still just noise? The second matters more than it sounds — ML-08
measured the baseline's precision@50 moving between 0.76 and 0.92 purely from how
ties were broken, so anything inside that band is not a result.


In [9]:
# --- monitoring and retrain triggers --------------------------------------
# Thresholds are set from measurements in this repo, so each one has a receipt.
# Where a number is a judgement call rather than a measurement, it says so.
TIE_LO, TIE_HI = 0.76, 0.92   # ML-08: baseline P@50 range from tie-breaking alone
HONEST_P50 = p_at_k(queue, 50)

triggers = pd.DataFrame([
    dict(signal="precision@50 on a fresh grouped split",
         trigger=f"drops below {BASE_RATE:.2f}",
         why="that is the base rate -- below it the queue is worse than picking at random",
         source="measured, this notebook"),
    dict(signal="share of top-50 that are zero-click pages",
         trigger="above 10%",
         why="the known failure mode reappearing; means the gate stopped working",
         source="measured, ML-08 and ML-09"),
    dict(signal="eligible page count",
         trigger="moves more than 20% month over month",
         why="the population changed, so the trained thresholds no longer describe it",
         source="judgement call, not measured"),
    dict(signal="feature missingness by content_type",
         trigger="any type crosses 40% missing",
         why="missingness tracks content_type, so a shift silently injects a category signal",
         source="data dictionary, measured in ML-05"),
    dict(signal="reviewer agreement",
         trigger="writers reject more than 1 in 3 queued pages",
         why="the cheapest drift detector in the system, and the only one that sees the page",
         source="judgement call, needs a feedback form that does not exist yet"),
])
print(triggers.to_string(index=False))
print()
print(f"Retrain cadence: quarterly, or on any trigger above. Not monthly -- the label is cut from a")
print(f"90-day window, so consecutive monthly retrains would train on largely the same rows.")
print()
print(f"Noise floor to respect: ML-08 measured the baseline's P@50 swinging {TIE_LO}-{TIE_HI} on")
print(f"tie-breaking alone. Today's queue sits at {HONEST_P50:.3f}. A change smaller than that band")
print("is not a result, and should not trigger anything.")

                                   signal                                      trigger                                                                            why                                                        source
    precision@50 on a fresh grouped split                             drops below 0.55    that is the base rate -- below it the queue is worse than picking at random                                       measured, this notebook
share of top-50 that are zero-click pages                                    above 10%             the known failure mode reappearing; means the gate stopped working                                     measured, ML-08 and ML-09
                      eligible page count         moves more than 20% month over month        the population changed, so the trained thresholds no longer describe it                                  judgement call, not measured
      feature missingness by content_type                 any type crosses 40% missing m

In [10]:
# --- cost and value -------------------------------------------------------
# The point is not to claim a revenue number. It is to show the arithmetic a
# manager would do, with the guesses labelled as guesses.
HOURS_PER_REFRESH = 3.0        # assumption, not measured here
K = 50

p = p_at_k(queue, K)
true_finds = p * K
random_finds = BASE_RATE * K

print(f"Reviewing the top {K} pages costs about {K * HOURS_PER_REFRESH:.0f} writer-hours")
print(f"  (assumption: {HOURS_PER_REFRESH} hours per page -- nothing here measures that)\n")
print(f"  from this queue     : {true_finds:.1f} of {K} are genuinely declining  (precision {p:.3f})")
print(f"  from a random {K}    : {random_finds:.1f} of {K}                        (base rate {BASE_RATE:.3f})")
print(f"  difference          : {true_finds - random_finds:.1f} more real finds for the same hours\n")
print("What that does and does not say:")
print("  - it says the queue spends a reviewer's time better than random order.")
print("  - it does NOT say those refreshes recover traffic. 'Declining' is the label;")
print("    'a refresh helps' is a different claim needing an experiment nobody has run.")
print("  - the honest next step is cheap: refresh a random half of the queued pages, hold")
print("    the other half, and compare after 90 days. That turns decision-support into evidence.")

Reviewing the top 50 pages costs about 150 writer-hours
  (assumption: 3.0 hours per page -- nothing here measures that)

  from this queue     : 44.0 of 50 are genuinely declining  (precision 0.880)
  from a random 50    : 27.5 of 50                        (base rate 0.551)
  difference          : 16.5 more real finds for the same hours

What that does and does not say:
  - it says the queue spends a reviewer's time better than random order.
  - it does NOT say those refreshes recover traffic. 'Declining' is the label;
    'a refresh helps' is a different claim needing an experiment nobody has run.
  - the honest next step is cheap: refresh a random half of the queued pages, hold
    the other half, and compare after 90 days. That turns decision-support into evidence.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The queue CSV is regenerated rather than committed: `work/**/*.csv` is
in `.gitignore` on purpose, since a ranked list of client pages is derived client
data. The figures and the metrics JSON do get committed — those are the receipts
the paper's numbers have to trace back to.


In [11]:
# --- exports the paper will build on --------------------------------------
# The queue CSV is deliberately gitignored (work/**/*.csv) -- it is derived client
# data. This notebook regenerates it. Figures and the metrics JSON are committed,
# because those are the receipts the paper's numbers trace back to.
EXPORT_COLS = ["queue_rank", "content_id", "client_id", "model_score", "archetype",
               "reason_code", "action", "why", "review_status", "gates_passed", "evidence",
               "impressions_90d", "clicks_90d", "ctr", "tier_median_ctr", "avg_position",
               "days_since_last_update", "word_count", "is_declining"]

queue_out = queue[EXPORT_COLS]
queue_path = OUT / "w07_action_queue.csv"
queue_out.to_csv(queue_path, index=False)

top200_path = OUT / "w07_action_queue_top200.csv"
queue_out.head(200).to_csv(top200_path, index=False)

metrics = {
    "split": "GroupShuffleSplit on client_id, test_size=0.3, random_state=42",
    "held_out_clients": int(queue.client_id.nunique()),
    "held_out_pages": int(len(queue)),
    "base_rate": round(BASE_RATE, 3),
    "roc_auc": round(float(roc_auc_score(y[te], scores)), 3),
    "pr_auc": round(float(average_precision_score(y[te], scores)), 3),
    "precision_at_k": {int(r.K): r.precision_at_K for r in prec.itertuples()},
    "lift_at_50": round(p_at_k(queue, 50) / BASE_RATE, 2),
    "archetype_counts": {k: int(v) for k, v in queue.archetype.value_counts().items()},
    "review_status_counts": {k: int(v) for k, v in queue.review_status.value_counts().items()},
    "decay_by_staleness": {str(i): {"pages": int(r.pages), "declining_rate": float(r.declining_rate)}
                           for i, r in decay.iterrows()},
    "decay_insight": "negative result - days_since_last_update is near-constant within client "
                     "(median 3 distinct values per client), range 4-301 days, so content decay is not "
                     "measurable in this snapshot",
    "no_go_count": len(NO_GO),
    "tie_noise_band_p50": [TIE_LO, TIE_HI],
    "assumption_hours_per_refresh": HOURS_PER_REFRESH,
}
metrics_path = OUT / "w07_playbook_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

for pth in (queue_path, top200_path, metrics_path,
            FIG / "w07_decay_by_staleness.png", FIG / "w07_precision_at_k.png"):
    print(f"  {pth}  ({pth.stat().st_size:,} bytes)")

  ..\outputs\w07_action_queue.csv  (1,133,488 bytes)
  ..\outputs\w07_action_queue_top200.csv  (80,933 bytes)
  ..\outputs\w07_playbook_metrics.json  (1,344 bytes)
  ..\figures\w07_decay_by_staleness.png  (57,314 bytes)
  ..\figures\w07_precision_at_k.png  (45,890 bytes)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [12]:
# --- self-check -----------------------------------------------------------
# Guards at the point where a mistake would be expensive: a leaked id, a claim
# with no receipt, or an export that silently wrote nothing.
import re
checks = []

def chk(name, ok, detail=""):
    checks.append((name, bool(ok), detail))

chk("every export exists and is non-empty",
    all(p.exists() and p.stat().st_size > 0 for p in
        [queue_path, top200_path, metrics_path, FIG / "w07_decay_by_staleness.png", FIG / "w07_precision_at_k.png"]))

chk("no client leaked across the split",
    not (set(elig.iloc[tr].client_id) & set(elig.iloc[te].client_id)))

chk("no label-derived column used as a feature", not (set(FEATURES) & BANNED))

# Ids are pseudonyms, but they are still derived client data. They belong in the
# ignored CSV, never in output that gets committed -- so check that git really is
# ignoring the file this notebook just wrote, rather than trusting the rule exists.
# First version of this check failed, and the check was the thing that was wrong:
# it handed git the path "../outputs/w07_action_queue.csv" -- relative to this
# notebook -- while running git from the repo root two folders up. git resolved
# that against the wrong directory and reported a file that does not exist as
# "not ignored". Absolute path, and let git find its own root.
import subprocess
REPO = Path("../..").resolve()
ignored = subprocess.run(["git", "check-ignore", "-q", str(queue_path.resolve())],
                         cwd=REPO, capture_output=True).returncode == 0
chk("the exported queue CSV is git-ignored", ignored, str(queue_path.name))

# Nothing printed anywhere in this notebook should carry an id. An earlier version
# of this check looked only at the queue preview and passed while a different cell
# printed client pseudonyms in a table index. Scan every output instead.
ID = re.compile(r"(content|client)_[0-9a-fA-F]{6,}")
nb_path = Path("w07_action_playbook.ipynb")
hits = []
if nb_path.exists():
    doc = json.loads(nb_path.read_text(encoding="utf-8"))
    for cell in doc["cells"]:
        for o in cell.get("outputs", []):
            hits += ID.findall("".join(o.get("text", "")))
chk("no pseudonymous ids in any committed notebook output", not hits,
    f"{len(hits)} found" if hits else "")

chk("every archetype has an action and a reason",
    set(queue.archetype.unique()) <= set(ACTIONS))

chk("the no-go class is blocked, not just labelled",
    (queue.loc[queue.archetype == "zero_click_ghost", "review_status"] == "BLOCKED - no-go").all())

chk("precision@50 is reported next to its base rate",
    "base_rate" in metrics and 50 in metrics["precision_at_k"])

chk("the queue beats the base rate at K=50",
    metrics["precision_at_k"][50] > metrics["base_rate"],
    f"{metrics['precision_at_k'][50]} vs {metrics['base_rate']}")

for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))

assert all(ok for _, ok, _ in checks), "a self-check failed -- fix it before committing"
print(f"\nall {len(checks)} checks passed")

  [PASS] every export exists and is non-empty
  [PASS] no client leaked across the split
  [PASS] no label-derived column used as a feature
  [PASS] the exported queue CSV is git-ignored  (w07_action_queue.csv)
  [PASS] no pseudonymous ids in any committed notebook output
  [PASS] every archetype has an action and a reason
  [PASS] the no-go class is blocked, not just labelled
  [PASS] precision@50 is reported next to its base rate
  [PASS] the queue beats the base rate at K=50  (0.88 vs 0.551)

all 9 checks passed
